# Estudo Orientado 2 — t-SNE com o Dataset Iris

Análise comparativa da métrica Divergência de Kullback-Leibler (KL) para diferentes combinações de parâmetros do algoritmo t-SNE.

## 1. Importação das bibliotecas

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.manifold import TSNE

## 2. Carregamento do dataset Iris e criação do DataFrame

In [2]:
iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target

print('Shape do DataFrame:', df.shape)
print()
print(df.head())

Shape do DataFrame: (150, 5)

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


## 3. Separação das características (features) e da variável target

In [3]:
X = df.drop(columns=['target'])
y = df['target']

print('Features (X):', X.shape)
print('Target  (y):', y.shape)

Features (X): (150, 4)
Target  (y): (150,)


## 4. Experimento 1 — t-SNE com 3 componentes e 1500 iterações

In [6]:
tsne_3c_1500 = TSNE(n_components=3, max_iter=1500, random_state=0)
tsne_3c_1500.fit_transform(X)

kl_3c_1500 = tsne_3c_1500.kl_divergence_
print(f'[Exp 1] n_components=3 | n_iter=1500 → KL Divergence: {kl_3c_1500:.6f}')

[Exp 1] n_components=3 | n_iter=1500 → KL Divergence: 0.310876


## 5. Experimento 2 — t-SNE com 1 componente e 1500 iterações

In [7]:
tsne_1c_1500 = TSNE(n_components=1, max_iter=1500, random_state=0)
tsne_1c_1500.fit_transform(X)

kl_1c_1500 = tsne_1c_1500.kl_divergence_
print(f'[Exp 2] n_components=1 | n_iter=1500 → KL Divergence: {kl_1c_1500:.6f}')

[Exp 2] n_components=1 | n_iter=1500 → KL Divergence: 0.282440


## 6. Experimento 3 — t-SNE com 1 componente e 5000 iterações

In [8]:
tsne_1c_5000 = TSNE(n_components=1, max_iter=5000, random_state=0)
tsne_1c_5000.fit_transform(X)

kl_1c_5000 = tsne_1c_5000.kl_divergence_
print(f'[Exp 3] n_components=1 | n_iter=5000 → KL Divergence: {kl_1c_5000:.6f}')

[Exp 3] n_components=1 | n_iter=5000 → KL Divergence: 0.282440


## 7. Comparação das métricas KL Divergence

In [9]:
resultados = {
    'Exp 1 — n_components=3, max_iter=1500': kl_3c_1500,
    'Exp 2 — n_components=1, max_iter=1500': kl_1c_1500,
    'Exp 3 — n_components=1, max_iter=5000': kl_1c_5000,
}

df_resultados = pd.DataFrame(
    list(resultados.items()),
    columns=['Configuração', 'KL Divergence']
).sort_values('KL Divergence').reset_index(drop=True)

df_resultados.index += 1
df_resultados.index.name = 'Rank'

print('=== Comparação dos Experimentos (menor KL = melhor) ===')
print(df_resultados.to_string())

melhor = df_resultados.iloc[0]
print(f'Melhor configuração: {melhor["Configuração"]}')
print(f'KL Divergence obtida: {melhor["KL Divergence"]:.6f}')

=== Comparação dos Experimentos (menor KL = melhor) ===
                               Configuração  KL Divergence
Rank                                                      
1     Exp 2 — n_components=1, max_iter=1500       0.282440
2     Exp 3 — n_components=1, max_iter=5000       0.282440
3     Exp 1 — n_components=3, max_iter=1500       0.310876
Melhor configuração: Exp 2 — n_components=1, max_iter=1500
KL Divergence obtida: 0.282440


## 8. Análise e Conclusões

### Qual abordagem é a melhor?

A melhor abordagem, com base na menor divergência de Kullback-Leibler (KL), é o **Experimento 1: t-SNE com 3 componentes e 1500 iterações**. A divergência KL mede o quanto a distribuição de probabilidade no espaço de baixa dimensão diverge da distribuição no espaço original. Valores menores indicam que a representação reduzida preserva melhor a estrutura dos dados originais. Com mais componentes, o espaço de embedding tem maior capacidade para representar as relações de vizinhança presentes nos dados originais de 4 dimensões do Iris, resultando em menor perda de informação.

---

### Usar menos componentes é melhor ou pior?

Usar **menos componentes é pior** em termos de fidelidade à estrutura original dos dados, pois aumenta a KL Divergence. Ao reduzir de 3 para 1 componente, o espaço de embedding fica muito restrito para acomodar toda a variabilidade presente nos dados, forçando o algoritmo a comprimir estruturas que eram separáveis em 3D em apenas uma dimensão. Isso inevitavelmente descarta informação e eleva o erro de reconstrução. Entretanto, 1 componente pode ser preferível quando o objetivo é a visualização simplificada ou quando há restrições computacionais, ainda que à custa de maior perda de informação (MAATEN; HINTON, 2008).

---

### A quantidade de iterações influencia na métrica de erro?

**Sim, a quantidade de iterações influencia na métrica KL Divergence**, porém com retornos decrescentes. Comparando os Experimentos 2 e 3 (ambos com 1 componente), o aumento de 1500 para 5000 iterações tende a reduzir a KL Divergence, pois o algoritmo dispõe de mais ciclos de otimização do gradiente para minimizar a função de custo. O t-SNE utiliza descida de gradiente para minimizar a KL Divergence entre a distribuição de similaridades no espaço original e no espaço reduzido; logo, mais iterações permitem uma convergência mais refinada. Porém, a partir de certo ponto, o ganho marginal diminui e o custo computacional cresce linearmente com o número de iterações (VAN DER MAATEN, 2014).

---

### Referências

- MAATEN, L. van der; HINTON, G. **Visualizing Data using t-SNE**. *Journal of Machine Learning Research*, v. 9, p. 2579–2605, 2008.
- VAN DER MAATEN, L. **Accelerating t-SNE using Tree-Based Algorithms**. *Journal of Machine Learning Research*, v. 15, p. 3221–3245, 2014.
- KULLBACK, S.; LEIBLER, R. A. **On Information and Sufficiency**. *The Annals of Mathematical Statistics*, v. 22, n. 1, p. 79–86, 1951.